In [1]:
%pip install -q "transformers>=4.51,<5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from pathlib import Path
import random
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out = Path("lab_outputs")
out.mkdir(exist_ok=True)
print("Device:", device)

Device: cuda


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
def continue_text(prompt, *, sample=False, temperature=0.8, seed=42):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    options = {"do_sample": sample, "max_new_tokens": 45,
               "pad_token_id": tokenizer.eos_token_id}
    if sample:
        options.update(temperature=temperature, top_p=0.9)
    with torch.no_grad():
        tokens = model.generate(**inputs, **options)
    return tokenizer.decode(tokens[0, inputs.input_ids.size(1):],
                            skip_special_tokens=True).strip()

prompts = [
    "A story about a robot begins like this:",
    "In a school library, a tiny robot found a lost book. It",
]
decoding_rows = []
for prompt in prompts:
    print("\nPROMPT:", prompt)
    text = continue_text(prompt)
    print("GREEDY:", text)
    decoding_rows.append((prompt, "Greedy", text))
    for temp in (0.6, 1.0):
        for seed in (42, 43):
            text = continue_text(prompt, sample=True, temperature=temp, seed=seed)
            print(temp, seed, text)
            decoding_rows.append((prompt, f"{temp} and {seed}", text))


PROMPT: A story about a robot begins like this:
GREEDY: A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed
0.6 42 I was in my basement, and a guy came in, and he said, "Hey, we're going to be building a robot. Let's go, and we're going to be building a robot that's
0.6 43 A robot is a computer program that communicates with humans via a computer program. The robot's job is to create and control a computer program. The robot's job is to control a computer program.

That's
1.0 42 It was supposed to be something big, but it never happened. It was called a toy, but no one liked it because it wasn't the most interesting thing in the universe, even though it was a real thing. But
1.0 43 A robotic girl runs into a strange, alien world called "Rome."

Rome's "Sculpture of Chaos" is where she's found. She asks for a special artifact from t

In [5]:
import csv
print(f"Total outputs: {len(decoding_rows)}  (har ek ko 0-2 score do)\n")
for i, (p, d, t) in enumerate(decoding_rows, 1):
    print(f"{i:>2}. [{p[:28]}...] {d:<10} -> {t[:110]}")
with open(out / "gpt2_decoding_outputs.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["prompt", "decoding_and_seed", "output", "coherence_0_2", "relevance_0_2", "creativity_0_2"])
    w.writerows([(p, d, t, "", "", "") for p, d, t in decoding_rows])
print("\nSaved: lab_outputs/gpt2_decoding_outputs.csv")

Total outputs: 10  (har ek ko 0-2 score do)

 1. [A story about a robot begins...] Greedy     -> A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to d
 2. [A story about a robot begins...] 0.6 and 42 -> I was in my basement, and a guy came in, and he said, "Hey, we're going to be building a robot. Let's go, and 
 3. [A story about a robot begins...] 0.6 and 43 -> A robot is a computer program that communicates with humans via a computer program. The robot's job is to crea
 4. [A story about a robot begins...] 1.0 and 42 -> It was supposed to be something big, but it never happened. It was called a toy, but no one liked it because i
 5. [A story about a robot begins...] 1.0 and 43 -> A robotic girl runs into a strange, alien world called "Rome."

Rome's "Sculpture of Chaos" is where she's fou
 6. [In a school library, a tiny ...] Greedy     -> was a book that had been lost for years.

"I was like, 'Oh my God, this is so weird,'" s

In [6]:
examples = [
    ("graphs", "Sketch a graph, then mark a path through three vertices."),
    ("Python loops", "Trace a loop by writing the counter after each step."),
    ("fractions", "Draw equal parts and label the fraction you shaded."),
    ("sets", "Draw a Venn diagram and label the intersection."),
    ("sorting", "Sort five values by hand and record each swap."),
    ("probability", "List the outcomes of two coin tosses before counting."),
]

tuned = AutoModelForCausalLM.from_pretrained(model_id).to(device)

tuned.requires_grad_(False)
tuned.transformer.h[-1].requires_grad_(True)
opt = torch.optim.AdamW(tuned.transformer.h[-1].parameters(), lr=2e-5)

for epoch in range(3):
    tuned.train()
    for topic, answer in examples:
        prefix = tokenizer.encode(f"Topic: {topic}\nStudy tip:")
        target = tokenizer.encode(" " + answer) + [tokenizer.eos_token_id]
        ids = torch.tensor([prefix + target], device=device)
        labels = torch.tensor([[-100] * len(prefix) + target], device=device)
        opt.zero_grad()
        loss = tuned(input_ids=ids, labels=labels).loss
        loss.backward()
        opt.step()
    print("Epoch:", epoch + 1, "last example loss:", round(loss.item(), 3))

tuned.save_pretrained(out / "study_tip_gpt2")
tokenizer.save_pretrained(out / "study_tip_gpt2")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch: 1 last example loss: 4.759
Epoch: 2 last example loss: 4.328
Epoch: 3 last example loss: 4.724


('lab_outputs/study_tip_gpt2/tokenizer_config.json',
 'lab_outputs/study_tip_gpt2/special_tokens_map.json',
 'lab_outputs/study_tip_gpt2/vocab.json',
 'lab_outputs/study_tip_gpt2/merges.txt',
 'lab_outputs/study_tip_gpt2/added_tokens.json',
 'lab_outputs/study_tip_gpt2/tokenizer.json')

In [7]:
held_out = ["recursion", "matrices", "binary search"]
base_model = model
heldout_rows = []
for topic in held_out:
    prompt = f"Topic: {topic}\nStudy tip:"
    model = base_model
    b = continue_text(prompt)
    print(topic, "base:", b)
    model = tuned.eval()
    t = continue_text(prompt)
    print(topic, "tuned:", t)
    heldout_rows.append((topic, b, t))
model = base_model

with open(out / "gpt2_heldout_outputs.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["topic", "base_output", "tuned_output", "base_score_4", "tuned_score_4"])
    w.writerows([(a, b, c, "", "") for a, b, c in heldout_rows])
print("\nSaved: lab_outputs/gpt2_heldout_outputs.csv")

recursion base: Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int i) { return i; } public: Recursion(int i)
recursion tuned: Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int x, int y) { return x + y; } public: Rec
matrices base: Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
matrices tuned: Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
binary search base: Use the search box to find the search term you want to search for.

Study tip: Use the search box to find the search term you want to search for.

Study tip: Use the search box to
binary search tuned: Use the search box to find the search